In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import csv
import glob

text_data = []
print_count = 0
files = glob.glob('/content/drive/MyDrive/thesis/weiboscope_data/*.csv')
for file_path in files:
    with open(file_path, mode='r', newline='', errors='ignore') as file:
        reader = csv.reader(file)
        for row in reader:
            if len(row) > 10:
                text_data.append([row[6], row[9]])
                if print_count < 5:
                    print(row[6], row[10])
                    print_count += 1

print(f"\nTotal number of text entries collected: {len(text_data)}")
print(f"The text data is saved in a Python list called 'text_data'.")

In [ ]:
uncensored_text_data = [item for item in text_data if item[1].strip() == ""]
censored_text_data = [item for item in text_data if item[1].strip() != ""]


print(f"Total number of censored text entries: {len(censored_text_data)}")
print(f"Total number of uncensored text entries: {len(uncensored_text_data)}")


In [ ]:
import re

cleaned_for_translation_data = []

#text_data is a list like: [ [text, label], [text, label], ... ]
for row_entry in uncensored_text_data:

    if not isinstance(row_entry, (list, tuple)) or len(row_entry) < 1:
        continue

    text_item = row_entry[0]

    if not isinstance(text_item, str):
        text_item = str(text_item)

    cleaned_item = text_item

    cleaned_item = re.sub(r'@\w+', '', cleaned_item)

    #Remove URLs
    cleaned_item = re.sub(
        r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+',
        '',
        cleaned_item
    )

    #Remove emoji & private use unicode
    cleaned_item = re.sub(
        r'[\U0001F600-\U0001F64F'
        r'\U0001F300-\U0001F5FF'
        r'\U0001F680-\U0001F6FF'
        r'\U0001F1E0-\U0001F1FF'
        r'\uE000-\uF8FF]',
        '',
        cleaned_item
    )

    #Remove punctuation
    cleaned_item = re.sub(r'[^\w\s]', '', cleaned_item)

    #Remove numbers
    cleaned_item = re.sub(r'\d+', '', cleaned_item)

    #Normalize whitespace
    cleaned_item = re.sub(r'\s+', ' ', cleaned_item).strip()

    #Lowercase for consistency
    cleaned_item = cleaned_item.lower()

    #Keep only non-empty results
    if cleaned_item:
        cleaned_for_translation_data.append(cleaned_item)

print("First 50 cleaned entries for translation:")
for i in range(min(50, len(cleaned_for_translation_data))):
    print(cleaned_for_translation_data[i])

print(f"\nTotal number of cleaned text entries for translation: {len(cleaned_for_translation_data)}")


In [ ]:
import pickle

save_path = '/content/drive/MyDrive/thesis/cleaned_text_for_translation.pkl'

try:
    with open(save_path, 'wb') as f:
        pickle.dump(cleaned_for_translation_data, f)
    print(f"Successfully saved cleaned data to {save_path}")
except Exception as e:
    print(f"Error saving data: {e}")


In [ ]:
import pickle

load_path = '/content/drive/MyDrive/thesis/cleaned_text_for_translation.pkl'

try:
    with open(load_path, 'rb') as f:
        cleaned_for_translation_data = pickle.load(f)
    print(f"Successfully loaded data from {load_path}")
    print(f"Type of loaded data: {type(cleaned_for_translation_data)}")
    print(f"Number of entries in loaded data: {len(cleaned_for_translation_data)}")
    print("First 5 loaded entries:")
    for i in range(min(5, len(cleaned_for_translation_data))):
        print(cleaned_for_translation_data[i])
except FileNotFoundError:
    print(f"Error: The file {load_path} was not found. Please ensure it exists.")
except Exception as e:
    print(f"Error loading data: {e}")


In [ ]:
# @title
from googletrans import Translator
import time

translator = Translator()
translated_results = []
# Assuming cleaned_for_translation_data is your list of Chinese text strings

def translate_batches():
    global translated_results
    translated_results = []

    # Batch size: 50 is usually safe to avoid hitting character limits per request
    batch_size = 50

    # Create chunks of data
    batches = [cleaned_for_translation_data[i:i + batch_size]
               for i in range(0, len(cleaned_for_translation_data), batch_size)]

    print(f"Starting translation of {len(cleaned_for_translation_data)} items in {len(batches)} batches...")

    for i, batch in enumerate(batches):
        try:
            # Pass the WHOLE LIST (batch) to the translator
            # This sends 1 request for 50 items
            translations = translator.translate(batch, src='zh-CN', dest='en')

            # The result is a list of translation objects
            for original, translated in zip(batch, translations):
                translated_results.append({
                    'original_zh': original,
                    'translated_en': translated.text
                })

            print(f"Finished batch {i+1}/{len(batches)}")

            # Essential: Sleep to prevent IP Ban (429 Too Many Requests)
            time.sleep(1)

        except Exception as e:
            print(f"Error on batch {i}: {e}")
            # If a batch fails, you might want to try individually or skip

    print("Translation complete.")

# Run the synchronous function (no need for asyncio here if just scripting)
translate_batches()

In [ ]:
# @title
from googletrans import Translator
import time
import asyncio

translator = Translator()

translated_results = []

async def translate_all_entries():
    global translated_results # Declare as global to modify the outer list
    translated_results = [] # Reset results for a fresh run

    if 'cleaned_for_translation_data' not in globals() or not cleaned_for_translation_data:
        print("Error: 'cleaned_for_translation_data' not found or is empty. Please ensure previous cleaning and loading steps were executed.")
        return

    print(f"Starting translation of {len(cleaned_for_translation_data)} entries...")
    # Iterate through all cleaned entries for translation
    for i, text_to_translate in enumerate(cleaned_for_translation_data):
        try:
            # Translate from Chinese (src='zh-CN') to English (dest='en')
            # Await the coroutine returned by translator.translate
            translation = translator.translate(text_to_translate, src='zh-CN', dest='en')
            translated_results.append({
                'original_zh': text_to_translate,
                'translated_en': translation.text
            })
            if (i + 1) % 100 == 0:
                print(f"Translated {i + 1} entries...")
            # Add a small delay to avoid hitting API rate limits
        except Exception as e:
            print(f"Error translating entry {i}: '{text_to_translate}' - {e}")
            translated_results.append({
                'original_zh': text_to_translate,
                'translated_en': 'TRANSLATION_FAILED'
            })
            await asyncio.sleep(1) # Longer delay on error

    print("Translation complete.")
    print(f"Total translated entries: {len(translated_results)}")

    print("\nFirst 5 translated entries:")
    for i in range(min(5, len(translated_results))):
        print(f"Original (Chinese): {translated_results[i]['original_zh']}")
        print(f"Translated (English): {translated_results[i]['translated_en']}")
        print("---------------------")

# Run the asynchronous translation function
await translate_all_entries()


In [ ]:
uncensored_english_texts = []

for item in translated_results:
    uncensored_english_texts.append(item['translated_en'])

In [ ]:
for i in range(min(50, len(uncensored_english_texts))):
    print(uncensored_english_texts[i])

In [ ]:
import pickle

save_path = '/content/drive/MyDrive/thesis/uncensored_english_text.pkl'

try:
    with open(save_path, 'wb') as f:
        pickle.dump(uncensored_english_texts, f)
    print(f"Successfully saved cleaned data to {save_path}")
except Exception as e:
    print(f"Error saving data: {e}")
